# Cyberion Defense Labs — Telemetry Analysis & Threat Hunting Notebook
**Project Code:** CDL-DET-V2  
**Author:** Detection Engineering & Threat Intelligence Practice  
**Date:** 2026-10-08  

This Jupyter Notebook demonstrates ad-hoc telemetry inspection, statistical beaconing analysis, and adversary TTP clustering across Windows Security, Sysmon, and Zeek datasets.

In [1]:
import os
import json
from collections import Counter

# Locate datasets
base_dir = os.path.dirname(os.getcwd())
data_dir = os.path.join(base_dir, 'data')
attack_file = os.path.join(data_dir, 'attack_telemetry.json')
benign_file = os.path.join(data_dir, 'benign_telemetry.json')

with open(attack_file, 'r', encoding='utf-8') as f:
    attack_events = json.load(f)

with open(benign_file, 'r', encoding='utf-8') as f:
    benign_events = json.load(f)

print(f"Loaded {len(attack_events)} attack events and {len(benign_events)} benign events.")

## 1. Log Source Breakdown
Inspect the distribution of log sources across the captured attack dataset.

In [2]:
logsource_counts = Counter(e.get('logsource') for e in attack_events)
print("Attack Telemetry Log Sources:")
for src, count in logsource_counts.items():
    print(f"  - {src:<28}: {count} records")

## 2. Threat Hunt 01: C2 Beaconing Analytics
Evaluate outbound connection parameters for periodicity, anomalous ports, and C2 user agents.

In [3]:
c2_events = [e for e in attack_events if e.get('logsource') in ['zeek_http', 'windows_network_connection']]
print(f"Found {len(c2_events)} potential egress sessions:")
for ev in c2_events:
    dst = ev.get('id.resp_h') or ev.get('DestinationIp')
    port = ev.get('id.resp_p') or ev.get('DestinationPort')
    uri = ev.get('uri', 'N/A')
    ua = ev.get('user_agent', 'N/A')
    print(f"  Target: {dst}:{port} | URI: {uri} | UA: {ua}")

## 3. Threat Hunt 02: LOLBAS & Reconnaissance Activity
Identify Living-Off-The-Land binaries executed by non-standard principals.

In [4]:
lolbas = ['whoami.exe', 'systeminfo.exe', 'nltest.exe', 'net.exe', 'wmic.exe', 'certutil.exe']
proc_events = [e for e in attack_events if any(tool in e.get('Image', '').lower() for tool in lolbas)]

print(f"Identified {len(proc_events)} LOLBAS execution artifacts:")
for ev in proc_events:
    print(f"  Host: {ev.get('ComputerName')} | User: {ev.get('User')} | Command: {ev.get('CommandLine')}")

## 4. False-Positive Validation
Verify that benign enterprise background noise contains legitimate administrative indicators that are properly filtered.

In [5]:
print("Benign Baseline Sample Entities:")
for b in benign_events:
    print(f"  [{b.get('_id')}] {b.get('description')}")